In [1]:
#import
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import log_loss, accuracy_score, recall_score, f1_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

In [2]:
#load embeddings
embeddings = pd.read_csv("clinical_embeddings_16D.csv")

In [3]:
#define features & labels
embedding_cols = [f"embed_{i}" for i in range(1, 17)]
X = embeddings[embedding_cols].values
y = embeddings["BCR"].values
patient_ids = embeddings["patient_id"].values
embedding_fold = embeddings["fold"].values if "fold" in embeddings.columns else None

In [4]:
#5 fold cross validation
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

In [5]:
#define models
#logistic regression
log_reg_model = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=1000, random_state=42))
])
#random forest
rf_model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    class_weight="balanced"
)

models = {
    "logistic_regression": log_reg_model,
    "random_forest": rf_model
}

In [ ]:
#run and generate summary
summary_rows = []

for model_name, model in models.items():
    fold_metrics = []

    for fold, (train_idx, test_idx) in enumerate(skf.split(X, y), start=1):
        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]

        #fit model
        model.fit(X_train, y_train)

        #predictions
        y_train_pred = model.predict(X_train)
        y_test_pred = model.predict(X_test)

        #predicted probabilities for class 1
        y_train_prob = model.predict_proba(X_train)[:, 1]
        y_test_prob = model.predict_proba(X_test)[:, 1]

        #metrics for this fold
        fold_row = {
            "train_loss": log_loss(y_train, y_train_prob),
            "test_loss": log_loss(y_test, y_test_prob),
            "train_accuracy": accuracy_score(y_train, y_train_pred),
            "test_accuracy": accuracy_score(y_test, y_test_pred),
            "test_recall": recall_score(y_test, y_test_pred, zero_division=0),
            "test_f1": f1_score(y_test, y_test_pred, zero_division=0)
        }
        fold_metrics.append(fold_row)

    #convert fold results to dataframe
    fold_metrics_df = pd.DataFrame(fold_metrics)

    #summarize across folds
    summary_row = {
        "model": model_name,
        "train_loss_mean": fold_metrics_df["train_loss"].mean(),
        "train_loss_sd": fold_metrics_df["train_loss"].std(),
        "test_loss_mean": fold_metrics_df["test_loss"].mean(),
        "test_loss_sd": fold_metrics_df["test_loss"].std(),
        "train_accuracy_mean": fold_metrics_df["train_accuracy"].mean(),
        "train_accuracy_sd": fold_metrics_df["train_accuracy"].std(),
        "test_accuracy_mean": fold_metrics_df["test_accuracy"].mean(),
        "test_accuracy_sd": fold_metrics_df["test_accuracy"].std(),
        "test_recall_mean": fold_metrics_df["test_recall"].mean(),
        "test_recall_sd": fold_metrics_df["test_recall"].std(),
        "test_f1_mean": fold_metrics_df["test_f1"].mean(),
        "test_f1_sd": fold_metrics_df["test_f1"].std()
    }

    summary_rows.append(summary_row)

#final summary
summary_df = pd.DataFrame(summary_rows)
print(summary_df)
summary_df.to_csv("clinical_model_summary_stats.csv", index=False)

                 model  train_loss_mean  train_loss_sd  test_loss_mean  \
0  logistic_regression         0.342908       0.032668        1.460261   
1        random_forest         0.227338       0.014163        0.928755   

   test_loss_sd  train_accuracy_mean  train_accuracy_sd  test_accuracy_mean  \
0      0.301482               0.9125           0.055902                0.35   
1      0.240748               1.0000           0.000000                0.30   

   test_accuracy_sd  test_recall_mean  test_recall_sd  test_f1_mean  \
0          0.136931               0.1        0.223607           0.1   
1          0.209165               0.1        0.223607           0.1   

   test_f1_sd  
0    0.223607  
1    0.223607  
